# 0. Page de garde

**Nom Prénom** : ALBURQUERQUE Julien  
**Formation** : CISIA — Concevoir et implémenter une solution d'intelligence artificielle (référentiel C1→C9)  
**Cas d'usage** : Détection précoce du décrochage étudiant en L1  
**Dépôt GitHub** : [cisia-decrochage-etudiant](https://github.com/jalb-code/cisia-decrochage-etudiant.git)  
**Date de soutenance** : 2026-09-01 — **Version du notebook** : v0.2 (2026-08-07)  
**Environnement** : Python 3.13 · `uv` (environnement figé par `uv.lock`) · scikit-learn — détail exhaustif en §15  

## _Scaffolding du projet_

- **Création de la structure du projet** — arborescence normalisée : code réutilisable dans `src/`, notebook unique, données hors dépôt, décisions tracées au **journal de bord** de chaque section
- **Hygiène de code** : dépôt normalisé, format/lint (`ruff`), tests (`pytest`), `pre-commit` et intégration continue (GitHub Actions) → un livrable reproductible et vérifiable.

### _Journal de bord des décisions_

1. **Ne pas versionner les données.**

   - _Question_ : Les jeux de données doivent-ils entrer dans le dépôt ?  
   - _Ce que j'ai regardé_ :  données scolaires nominatives à faible volume ; dépôt rendu temporairement public pendant la certification ; irréversibilité d'un fichier entré dans l'historique Git.  
   - _Décision_ : rien n'entre dans le dépôt : `data/` reste hors Git (arborescence tenue par des `.gitkeep`), approvisionnement documenté (`README.md`, `data/README.md`).   
   - _Pourquoi_ : Précaution RGPD (minimisation), au prix assumé d'un clone non exécutable tel quel.  

2. **Usage et gouvernance de l'assistance IA.**
    - _Question_ : À quelles conditions m'appuyer sur l'assistance IA (Claude Code) pour un livrable censé démontrer *mes* compétences, de façon défendable et transparente ?
    - _Ce que j'ai regardé_ : l'IA générative fait partie de la pratique courante du métier ; risque de déléguer implicitement la décision à l'IA; exigence d'auditabilité et de posture éthique tenable devant le jury. 
    - _Décision_ : socle IA cadré + *human-in-the-loop* : Claude Code code sous relecture, l'humain décide et assume ; trois relecteurs en lecture seule (consultants ML et RGPD, gardien du cas d'usage) éclairent sans jamais décider ; chaque décision reste tracée dans le notebook
    - _Pourquoi_ : Garder l'humain décideur et les agents en lecture seule empêche toute décision autonome de l'IA, rend le travail auditable et la posture tenable devant le jury — l'usage de l'IA est assumé, non masqué.

# 1. Résumé exécutif

**Problème** : Dans une université pluridisciplinaire, ~28 % des étudiants de L1 de la cohorte abandonnent (1 479 sur 5 200). Aujourd'hui, l'abandon ne se constate qu'en fin de semestre — trop tard pour agir.

**Objectifs** : Concevoir une solution d'IA capable de prédire, **à mi-S1**, le risque de décrochage, afin de proposer un accompagnement au bon moment. 2 cibles de prédictions sont demandées: 
  - **Risque d'abandon** — score de risque (classification) : faut-il proposer un accompagnement ?
  - **Moyenne finale (/20)** — régression : calibrer et prioriser l'intensité de l'accompagnement.

## Approche

Une démarche de bout en bout, du besoin métier au service qui dure ; chaque étape est développée et **justifiée dans sa section** (journal de bord).

```mermaid
flowchart TD
    A["1 · Comprendre le besoin<br/>décrochage constaté trop tard → agir à mi-S1<br/>Section §2"]
    B["2 · Cadrer avant tout<br/>cibles · RGPD, éthique, responsabilités · aide à la décision<br/>Section §2–§4"]
    C["3 · Fiabiliser la donnée<br/>profilage · conformation · EDA<br/>exclusions de principe : anti-fuite, minimisation · features<br/>Section §5–§7"]
    D["4 · Modéliser et comparer<br/>baseline + ≥2 familles → départage sur AUC<br/>classification + régression<br/>Section §8–§9"]
    E["5 · Prouver et arbitrer<br/>métriques · seuil = coût FN&gt;FP · SHAP · audit d'équité<br/>Section §9, §12"]
    F["6 · Industrialiser<br/>pipeline sérialisé (score) · architecture cible<br/>Section §10–§11"]
    G["7 · Assumer les limites et durer<br/>limites connues · suivi de dérive · ré-entraînement · versioning<br/>Section §13–§14"]
    A --> B --> C --> D --> E --> F --> G
    G -.->|nouvelle promotion / dérive| C
```

## Résultats clés

- **Détection du décrochage (`abandon`)** : les modèles sont départagés sur l'**AUC = ⟨valeur — §12⟩** (indépendante du seuil) ; au seuil retenu, le modèle retrouve **⟨rappel en % — §12⟩** des décrocheurs.
- **Priorisation (`moyenne_finale`)** : régression de la note finale (**⟨RMSE/MAE — §12⟩**, R² = ⟨valeur — §12⟩), pour calibrer l'intensité de l'accompagnement.
- **Décision métier** : seuil fixé par l'**asymétrie des coûts** (un décrochage manqué coûte plus qu'un signalement à tort) : **⟨N — §12⟩** étudiants signalés, dont **⟨M — §12⟩** décrocheurs réels (précision ⟨% — §12⟩).
- **Explicabilité** : restitution **globale et locale** (SHAP)
- **Industrialisation** :  pipeline sérialisé + contrat `predict()`, architecture cible et suivi de dérive.

## Limites

- **Généralisation et dérive** : une seule cohorte (2024-2025, données synthétiques) et une validation intra-cohorte ⇒ le modèle est susceptible de devoir être **ré-entraîné** à la prochaine promotion.
- **Complétude des données** : manquants sur 10 colonnes ⇒ Mise en place de stratégie d'imputation comme parade à cette limite
- **Calibrage du seuil** : coûts métier (FN/FP) et capacité d'accompagnement non fournis ⇒ hypothèse de travail : un décrochage manqué (FN) pèse plus qu'un signalement à tort (FP), sans que ce dernier soit anodin.
- **Déployabilité conditionnée** : la mise en production dépend d'actes du responsable de traitement (AIPD, base légale confirmée par le DPO, information des étudiants) et d'effets de bord du marquage à documenter

*TODO — à compléter après §12 : cohorte unique et synthétique ; seuil à recalibrer par promotion.*

# 2. Cadrage métier et cas d'usage — journal de bord [C1]

## Problématique métier

Une université pluridisciplinaire constate un **taux d'abandon élevé en L1**. Aujourd'hui, l'abandon se **constate en fin de semestre** — trop tard pour agir.

La **direction de la réussite étudiante** veut identifier, **dès mi-parcours du S1**, les étudiants à risque de décrochage, pour déclencher un accompagnement (tutorat, soutien méthodologique, aide sociale).

## Enjeux métier

- **Humain** — un décrochage est souvent **réversible s'il est repéré tôt** ; sinon, démotivation et impact durable sur l'insertion professionnelle.
- **Institutionnel** — les ressources d'accompagnement sont **limitées** (heures de tutorat, aides) ⇒ il faut **cibler** et allouer là où c'est le plus utile. C'est le rôle de la cible secondaire : prioriser et calibrer l'intensité.
- **Économique** — un abandon est un investissement gaspillé, à un coût bien plus important que la mise en place d'un accompagnement.

## Objectifs et contraintes

Deux prédictions complémentaires à mi-S1 :

| Cible | Variable | Tâche | Rôle |
|---|---|---|---|
| **Principale** | `abandon` (0/1) | Classification binaire | Faut-il proposer un accompagnement ? |
| **Secondaire** | `moyenne_finale` (/20) | Régression | Prioriser et calibrer l'intensité de l'accompagnement |

La cible principale déclenche l'action ; la secondaire la **module**. La solution livre un **appui à la décision** — un score, non un verdict.

Trois contraintes pèsent sur la solution dès le cadrage et orientent toute la chaîne :

- **Horizon mi-S1 — risque de fuite temporelle.** La prédiction se fait à mi-parcours du S1 : une variable n'est légitime que si elle est **connue à cet instant**. Une donnée consolidée en fin de S1 gonflerait la performance en validation tout en rendant le modèle inexploitable en production. Reste à établir en **§3** quelles variables sont disponibles à mi-S1 ; toute colonne consolidée en fin de S1 est à écarter du périmètre de scoring.
- **Explicabilité exigée — restitution SHAP.** L'énoncé impose des indicateurs **explicables** ; la solution prévoit une restitution **SHAP**, globale et locale. Le choix et l'alternative écartée sont au journal de bord.
- **Éthique et RGPD — variables sensibles.** L'énoncé exige un usage **éthique et conforme** des données étudiantes. Points de vigilance à traiter en **§4 [C2]** : biais portés par les variables sensibles, et qualification de la décision au regard de l'art. 22.

## Le coût métier d'une erreur : décrochage manqué (FN) vs signalement à tort (FP)

Le modèle peut se tromper de **deux façons**, qui n'ont ni les mêmes conséquences ni le même coût :

- **Faux négatif (FN) — un décrochage manqué.** Le modèle prédit « pas d'abandon » pour un étudiant qui, en réalité, décroche : aucun accompagnement n'est proposé à qui en aurait eu besoin, l'occasion d'agir à temps est perdue. **Coût le plus lourd, et souvent irréversible** - une chance de réussite qui ne revient pas pour l'étudiant, une année d'investissement perdue pour l'institution.
- **Faux positif (FP) — un signalement à tort.** Le modèle prédit « à risque » pour un étudiant qui, en réalité, ne décroche pas : une ressource d'accompagnement limitée est mobilisée pour rien. **Coût borné et réversible** - un créneau de tutorat consommé en trop ; s'y ajoute, pour l'étudiant, une étiquette « à risque » qui n'est pas neutre.

Les deux coûts ne sont pas du même ordre : **un étudiant perdu pèse plus qu'un créneau de tutorat dépensé**. Restent ouverts, faute de montants : la **pondération** et le **seuil** (à arbitrer en **§12 [C8]**) ; et le **coût éthique du FP**, la stigmatisation (à traiter en **§4 [C2]**).

## Parties prenantes

| Acteur | Rôle | Ce qu'on doit lui fournir |
|---|---|---|
| **Direction réussite étudiante** | Commanditaire | Une solution répondant aux objectifs qu'elle a définis. |
| **Tuteurs / responsables pédagogiques** | Utilisateurs | Un outil qui, à mi-S1, signale le risque, estime la moyenne finale et **explique** la prédiction, pour adapter l'accompagnement. |
| **Étudiant** | Sujet des données **et** bénéficiaire | Minimisation des données ; vigilance à traiter en §4 sur les biais et la stigmatisation. |
| **DPO / référent éthique** | Garant de la conformité | Les livrables de conformité — à préciser en §4. |
| **SI scolarité / LMS** | Fournisseur des données | Un accès fiable et documenté ; intégration technique à préciser en §10–§11 [C6/C7]. |

## _Journal de bord des décisions_

1. **Coût métier inconnu, mais asymétrie évidente (FN ≫ FP).**
   - _Question_ : comment pondérer un décrochage manqué (FN) face à un signalement à tort (FP), et régler le seuil, sans la moindre information sur le coût métier ?
   - _Ce que j'ai regardé_ : les données ne portent **aucun montant** et la capacité d'accompagnement n'est pas fournie ; en revanche le **sens** de l'asymétrie est certain — un FN pèse plus qu'un FP (cf. coût métier ci-dessus).
   - _Décision_ : retenir l'**asymétrie qualitative** « un FN coûte plus qu'un FP » ; départager les modèles sur une métrique **indépendante du seuil** (ROC/AUC) ; reporter le réglage du seuil en **§12 [C8]**, où **plusieurs scénarios** seront étudiés.
   - _Pourquoi_ : sans montant ni capacité, aucun seuil ne se **déduit** par le calcul ; je m'appuie donc sur ce qui est certain — le sens de l'asymétrie — et sur une métrique que le seuil n'affecte pas, en renvoyant son calibrage là où il s'instruit vraiment (§12).

2. **L'explicabilité, une contrainte métier.**
   - _Question_ : l'explicabilité est une contrainte métier posée par l'énoncé — quel est son impact sur la solution et le choix du modèle ?
   - _Ce que j'ai regardé_ : l'énoncé l'**exige** (indicateurs explicables, SHAP cité en exemple) ; le besoin des équipes pédagogiques est double — **global** (quels facteurs pèsent sur le risque) et **local** (pourquoi *cet* étudiant est signalé, pour adapter l'accompagnement).
   - _Décision_ : fournir une explicabilité **globale et locale** (SHAP) ; et l'intégrer au choix du modèle.
   - _Pourquoi_ : un tuteur agit sur un cas précis — sans le « pourquoi » local, il ne peut pas calibrer l'accompagnement ; et une contrainte intégrée au choix du modèle en amont vaut mieux qu'une explication plaquée après coup.

# 3. Données : disponibilité, gouvernance et alternatives — journal de bord [C1]

vérifier disponibilité/accès ;
intégrer le catalogue des formations ; envisager des alternatives en cas de données manquantes.

# 4. Enjeux éthiques, sociétaux et conformité — journal de bord [C2]

# 5. Chargement et compréhension des données [C3]

**Objectif** : établir ce que les fichiers contiennent, comment c'est écrit et ce qui cloche - puis conformer les données afin de produire le palier **bronze** sur lequel portera l'EDA (§6).

> 🔧 **Comment je m'y prends** - Le profilage est produit par un **script Python** (`profiling.profile_csv`) qui déduit tout du contenu de chaque fichier : encodage, délimiteur, puis, colonne par colonne, type, motifs d'écriture, bornes, manquants et non-conformité. Il en écrit **un rapport HTML complet par fichier** qui me sert de support à mes constatations et mes décisions.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from decrochage_l1 import schema
from decrochage_l1.config import settings
from decrochage_l1.data import bronze, cleaning, profiling

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 38)

# `report_dir` déclenche l'écriture du rapport HTML complet - hors dépôt, car il cite
# des valeurs brutes. Sans lui, mesurer ne touche pas au disque.
profils = {
    chemin.name: profiling.profile_csv(chemin, report_dir=settings.report_dir)
    for chemin in sorted(settings.raw_dir.glob("*.csv"))
}
profil_catalogue, profil_etudiants = sorted(profils.values(), key=lambda profil: profil.file.n_rows)

# Liens vers les rapports HTML complets
liens = "\n".join(
    f"- [{profil.report_path.name}]"
    f"(../{profil.report_path.relative_to(settings.root_dir).as_posix()})"
    for profil in profils.values()
    if profil.report_path is not None
)
display(Markdown(f"**> Rapports de profilage générés**\n\n{liens}"))

**> Rapports de profilage générés**

- [profil-dataset_catalogue_formations_V5.html](../reports/profil-dataset_catalogue_formations_V5.html)
- [profil-dataset_decrochage_etudiants_complet_V5.html](../reports/profil-dataset_decrochage_etudiants_complet_V5.html)

## 5.1 Profilage des fichiers

In [2]:
# Une ligne par fichier : encodage et délimiteur sont déduits du contenu.
display(
    pd.concat(
        [profil.file.to_frame().set_index("Propriété").T for profil in profils.values()],
        ignore_index=True,
    )
    .rename_axis(columns=None)
    .set_index("Fichier")
)

,Encodage,Délimiteur,Lignes,Colonnes,Lignes en double,Taille
Fichier,,,,,,
dataset catalogue_formations_V5.csv,utf-8-sig (avec BOM),"« , »",8,7,0,0.5 Kio
dataset decrochage_etudiants_complet_V5.csv,utf-8-sig (avec BOM),"« , »",5 240,33,40,887.4 Kio


**Constat ⇒ Décision**

- **40 paires strictement identiques** sur les 33 colonnes ⇒ Supprimer les lignes en double

## 5.2 Profilage des colonnes et de leur contenu

> 🔎 **Où lire le détail complet.** L'`overview` ci-dessous donne les huit indicateurs qui tiennent à l'écran. Les six autres - motifs d'écriture, non-conformité, et surtout l'**inventaire exhaustif des écritures rencontrées, modalité par modalité** - sont dans le **rapport HTML** généré en tête de §5, mon support de travail pour ces constats.

**Constat ⇒ Décision**

- `filiere` porte 8 valeurs pour 8 lignes ⇒ c'est la clé primaire de la table et sera utilisée comme clé de jointure.
- `niveau` est **constant** (`L1`), donc sans information ⇒ exclusion de principe sur le **gold dataset**

**⇒ Rien à mettre en forme.**

**Constat ⇒ Décision**
- `annee_universitaire` constante (`2024-2025`) ⇒ variance nulle, **à exclure** dans le **gold dataset**
- **Erreurs de format ⇒ à normaliser** - *la valeur est juste, l'écriture non*
  - Chiffrées restées en texte : `date_inscription` (3 formats), `taux_presence_pct` (« % » + virgule), `distance_domicile_km` (« km » + virgule), `moyenne_partiels_s1` (virgule)
  - Catégorielles (casse / accents / espaces) : `filiere` (31→8), `bac_type` (12→7), `sexe` (11→8), `mention_bac` (11→8), `boursier` (8→6)
- **Valeurs hétérogènes ⇒ à recoder** - *plusieurs libellés, une même valeur*
  - `sexe` : `f`/`femme` · `h`/`m`/`homme` · `nb`/`autre` · `nr`
  - `bac_type` : `gen`/`general`/`generale` · `techno`/`technologique` · `pro`/`professionnel`
  - `mention_bac` : `p`/`passable` · `ab`/`assez bien` · `b`/`bien` · `tb`/`tres bien`
  - `boursier` : `non`/`n`/`0` · `oui`/`o`/`1`

**Cohérence à vérifier ⇒ à faire en §5.3**
- `student_id`, `id_dossier` : un même identifiant porte-t-il deux lignes **divergentes**, hors des 40 doublons exacts ?
- `nb_devoirs_rendus ≤ nb_devoirs_total` et `nb_ue_validees_s1 ≤ nb_ue_total` sur toutes les lignes ?
- `filiere` : **jointure** avec le catalogue réalisable ?

**Questions reportées ⇒ à traiter lors de l'EDA §6**
- Valeurs absentes - 10 colonnes, jusqu'à 49,29 % : mécanisme (refus, MAR…) à qualifier

## 5.3 Contrôles de cohérence

### 5.3.1 `student_id`, `id_dossier` : Identifiant unique hors des 40 lignes en double ?

In [5]:
etudiants = profil_etudiants.data
jumelles = etudiants.duplicated(keep=False)

for identifiant in ("student_id", "id_dossier"):
    conflits = int((etudiants[identifiant].duplicated(keep=False) & ~jumelles).sum())
    display(Markdown(f"- `{identifiant}` portant deux lignes divergentes : **{conflits}**"))

- `student_id` portant deux lignes divergentes : **0**

- `id_dossier` portant deux lignes divergentes : **0**

✅ **student_id** et **id_dossier** uniques dans le dataset - une fois les lignes en double supprimées, 1 ligne = 1 étudiant

### 5.3.2 `nb_devoirs_rendus ≤ nb_devoirs_total` et `nb_ue_validees_s1 ≤ nb_ue_total` sur toutes les lignes ?

In [6]:
mesurees = sorted({colonne for paire in schema.ORDER_CONSTRAINTS for colonne in paire})
nombres = profil_etudiants.data[mesurees].apply(cleaning.parse_number)

display(profiling.check_order_constraints(nombres, schema.ORDER_CONSTRAINTS))

,contrainte,n_comparables,n_non_comparables,n_violations
0,nb_devoirs_rendus <= nb_devoirs_total,5240,0,0
1,nb_ue_validees_s1 <= nb_ue_total,5240,0,0


✅ `nb_devoirs_rendus ≤ nb_devoirs_total` et `nb_ue_validees_s1 ≤ nb_ue_total` toujours vrai

### 5.3.3 `filiere` : **jointure** avec le catalogue réalisable ?

In [7]:
communes = sorted(set(profil_etudiants.data.columns) & set(profil_catalogue.data.columns))
cle = communes[0]

# Rapprochement sur les valeurs NORMALISÉES : « Gestion » et « GESTION » sont la même
# filière. Comparées telles qu'écrites, elles compteraient toutes deux pour orphelines.
etudiants_cle = cleaning.normalize_text(profil_etudiants.data[cle])
catalogue_cle = cleaning.normalize_text(profil_catalogue.data[cle])

reference = set(catalogue_cle)
appariables = etudiants_cle.isin(reference)
orphelines = sorted(set(etudiants_cle.dropna()) - reference)
taux = 100 * appariables.mean()

with pd.option_context("display.max_colwidth", 100):
    display(
        pd.DataFrame(
            {
                "Mesure": [
                    "Nb filière (Étudiants)",
                    "Nb filière (Catalogue)",
                    "Lignes appariables",
                    "Filières hors catalogue",
                ],
                "Valeur": [
                    str(etudiants_cle.nunique()),
                    str(catalogue_cle.nunique()),
                    f"{appariables.sum()} / {len(etudiants_cle)} ({taux:.1f} %)",
                    ", ".join(orphelines) if orphelines else "aucune",
                ],
            }
        ).set_index("Mesure")
    )

,Valeur
Mesure,
Nb filière (Étudiants),8
Nb filière (Catalogue),8
Lignes appariables,5240 / 5240 (100.0 %)
Filières hors catalogue,aucune


✅ 100 % d'appariement ⇒ La jointure est possible

## 5.4 Thématisation

Je procède à une classification par thème des colonnes afin de rendre leur lecture plus aisée dans le cas où je mettrai en place la thématisation dans l'explicabilité des features.

In [8]:
colonnes = profil_etudiants.data.columns

with pd.option_context("display.max_colwidth", 90):
    display(
        pd.DataFrame(
            [{"colonnes": ", ".join(membres)} for membres in schema.COLUMN_THEMES.values()],
            index=pd.Index(schema.COLUMN_THEMES, name="thème"),
        )
    )

display(
    Markdown(
        f"Classées : **{len(schema.theme_by_column())} / {len(colonnes)}** · "
        f"non classées : **{schema.unclassified(colonnes) or 'aucune'}** · "
        f"citées mais absentes : **{schema.unknown(colonnes) or 'aucune'}**"
    )
)

,colonnes
thème,
Identifiants,"student_id, id_dossier"
Contexte d'inscription,"annee_universitaire, filiere, date_inscription"
Profil social et démographique,"age, sexe, boursier, distance_domicile_km, heures_travail_remunere_sem"
Parcours antérieur,"bac_type, mention_bac, etablissement_origine"
Engagement LMS,"connexions_lms_30j, heures_lms_total, ressources_consultees, messages_forum"
Assiduité et travail rendu,"taux_presence_pct, retards_rendus, nb_devoirs_total, nb_devoirs_rendus"
Résultats académiques,"moyenne_partiels_s1, nb_ue_total, nb_ue_validees_s1"
Ressenti déclaré,"motivation, satisfaction, sentiment_appartenance"
Avis du tuteur,commentaire_tuteur


Classées : **33 / 33** · non classées : **aucune** · citées mais absentes : **aucune**

## 5.5 Production du palier bronze

Le fichier écrit dans `data/bronze/` est une **copie de contrôle**, inspectable dans un tableur.
Ce n'est pas ce que §6 consomme : les `DataFrame` rendus ci-dessous circulent d'une section à l'autre, avec les types que la conformation leur a donnés.

In [9]:
jeux, rapports = {}, []

for nom, fichier in bronze.SOURCES.items():
    profil_source = profiling.profile_csv(settings.raw_dir / fichier)
    jeux[nom], rapport = bronze.build(profil_source, settings.bronze_dir / f"{nom}.csv")
    rapports.append(rapport)

etudiants_bronze, catalogue_bronze = jeux["etudiants"], jeux["catalogue"]

display(bronze.summary(rapports))

,fichier,lignes_source,doublons_retires,lignes_bronze,colonnes,colonnes_recodees
0,etudiants.csv,5240,40,5200,33,4
1,catalogue.csv,8,0,8,7,0


✅ Palier **Bronze** construit

# 6. Analyse exploratoire (EDA) : visualisations et interprétation — journal de bord [C3]

**Objectif** : mesurer et interpréter le jeu bronze pour décider - cibles, manquants, variables à écarter - sans modifier aucune donnée ; les décisions s'appliquent au gold (§7) et au Pipeline (§9).

# 7. Préparation des données (nettoyage, manquants, transformations, features) — journal de bord [C3]

# 8. Choix du modèle et démarche scientifique (baseline, modèles, comparaison) — journal de bord [C4]

Contrainte : 
- AUC départage la performance prédictive mais PR-AUC, Rappel, Precision et F1 pertinent à comparer également
- L'explicabilité est un critère du modèle

# 9. Entraînement, validation et ajustement (sélection du modèle final) — journal de bord [C5]

# 10. Implémentation et mise en exploitation (déploiement, exemple d’usage) — journal de bord [C6]

# 11. Architecture cible et contraintes [C7]

# 12. Mesure de performance et impacts (métriques techniques + métier) [C8]

# 13. Amélioration continue (ré-entraînement, suivi, versioning) [C9]

# 14. Conclusion (synthèse et recommandations)


# 15. Annexes (versions, paramètres, dépendances, fonctions utilitaires)